# Stage 4 — Numerical Analysis (Regression Models)

For each model in `REGRESSION_MODELS`, trains and evaluates two variants
per seed:

- **Baseline** — numerical AMZN market/fundamental features.
- **+NLP** — baseline features plus rolling headline probability features
  from Stage 3 (`artifacts/nlp_probs.parquet`).

Uses a chronological split with the same shared cutoff as Stages 2 and 3.
Saves `./results/regression_results.csv`.


In [ ]:
# Load shared helpers/config plus model registries.
from common import *

# Regression metrics used for evaluation.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# inspect is used to check whether model factory expects a seed.
import inspect
# datetime is used to record wall-clock ISO timestamps for external energy join.
from datetime import datetime, timezone

# Load artifacts from previous stages.
text_df = load_text_df()
num_df = load_num_df()
# nlp_probs_df contains per-headline probabilities from Stage 3.
nlp_probs_df = load_nlp_probs_df()

# Normalize timestamps to date granularity for consistent splitting/joining.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# Shared chronological split boundary to align text and numeric evaluations.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)

# Build +NLP numerical table by attaching rolling sentiment probabilities.
num_aug_df = attach_nlp_prob_features(
    num_df=num_df,
    nlp_probs_df=nlp_probs_df,
    lookback_days=CONFIG["regression"]["nlp_lookback_days"],
    date_col="date",
)

# Collector for per-run records saved at end of stage.
results = ResultsCollector()

## 4.1 Training routine

In [ ]:
def train_regression_model(model_name, model_factory, X_tr, X_te, y_tr, y_te, seed):
    """Train one regression model for one seed and return metrics + fitted model."""

    # Some factories need a seed argument; others do not.
    sig = inspect.signature(model_factory)
    model = model_factory(seed) if sig.parameters else model_factory()

    # Measure fit time + wall-clock anchors for external energy join.
    wall_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    wall_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Measure prediction time + wall-clock anchors for external energy join.
    wall_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    y_pred = model.predict(X_te)
    infer_time = time.time() - t0
    wall_infer_end_iso = datetime.now(timezone.utc).isoformat()

    # mse is used directly and also converted to rmse.
    mse = mean_squared_error(y_te, y_pred)
    return {
        "model": model_name,
        "seed": seed,
        "mse": mse,
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(y_te, y_pred),
        "r2": r2_score(y_te, y_pred),
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_test_samples": len(y_te),
        "wall_train_start_iso": wall_train_start_iso,
        "wall_train_end_iso": wall_train_end_iso,
        "wall_infer_start_iso": wall_infer_start_iso,
        "wall_infer_end_iso": wall_infer_end_iso,
    }, model

## 4.2 Run every (model, seed) × (baseline, +NLP) combination

In [ ]:
# Optional model store for later diagnostics.
trained_reg_models = {}

# Baseline numerical split (no NLP features).
(
    X_tr_base,
    X_te_base,
    y_tr,
    y_te,
    _,
    base_feature_cols,
    _,
    _,
) = make_num_splits_chronological(
    df=num_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Augmented numerical split (includes rolling NLP probabilities).
(
    X_tr_aug,
    X_te_aug,
    y_tr_aug,
    y_te_aug,
    _,
    aug_feature_cols,
    _,
    _,
) = make_num_splits_chronological(
    df=num_aug_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Ensure both variants use identical targets and row ordering.
if not np.array_equal(y_tr, y_tr_aug) or not np.array_equal(y_te, y_te_aug):
    raise ValueError("Baseline and +NLP splits do not align on targets.")

print(f"Baseline feature count: {len(base_feature_cols)}")
print(f"Augmented feature count: {len(aug_feature_cols)}")

# Train every model for every seed for baseline and +NLP variants.
for model_name, factory in REGRESSION_MODELS.items():
    for seed in CONFIG["random_seeds"]:
        baseline_record, baseline_model = train_regression_model(
            model_name=model_name,
            model_factory=factory,
            X_tr=X_tr_base,
            X_te=X_te_base,
            y_tr=y_tr,
            y_te=y_te,
            seed=seed,
        )
        results.add_regression(baseline_record)
        trained_reg_models[(model_name, seed, "baseline")] = baseline_model

        augmented_record, augmented_model = train_regression_model(
            model_name=f"{model_name} +NLP",
            model_factory=factory,
            X_tr=X_tr_aug,
            X_te=X_te_aug,
            y_tr=y_tr,
            y_te=y_te,
            seed=seed,
        )
        results.add_regression(augmented_record)
        trained_reg_models[(model_name, seed, "+NLP")] = augmented_model

        # Print side-by-side R² so uplift is easy to inspect while running.
        print(
            f"{model_name} (seed {seed})  "
            f"baseline R²={baseline_record['r2']:.4f}, "
            f"+NLP R²={augmented_record['r2']:.4f}"
        )

## 4.3 Baseline vs +NLP comparison

In [ ]:
reg_df = results.regression_df()
# Mean baseline R² by base model name.
baseline_mean = reg_df[~reg_df["model"].str.endswith("+NLP")].groupby("model")["r2"].mean()
# Mean augmented R² by base model name (remove suffix for alignment).
aug_mean = reg_df[reg_df["model"].str.endswith("+NLP")].assign(
    base_model=lambda d: d["model"].str.replace(" +NLP", "", regex=False)
).groupby("base_model")["r2"].mean()
comparison = pd.concat(
    [baseline_mean.rename("R²_baseline"), aug_mean.rename("R²_+NLP")], axis=1
)
comparison["R²_improvement"] = comparison["R²_+NLP"] - comparison["R²_baseline"]
print("=== Baseline vs NLP-Augmented R² (mean over seeds) ===")
display(comparison.round(6))
reg_df.round(6)

## 4.4 Persist results

In [ ]:
results.save(RESULTS_DIR)
print(f"Total regression experiments: {len(results.regression_results)}")